In [ ]:
# ============================================================
# fix_pass2_npl_and_review.ipynb
#
# WHAT THIS DOES IN ONE SESSION:
#
# PART A - NPL/Coverage/LCR for all 63 bank records
#   Problem: NPL ratios appear in appendix pages (15-50),
#   far beyond the 8-12 baseline pages we sent before.
#   Fix: scan ALL pages of each bank PDF for NPL keywords,
#   extract text from only those 1-3 matching pages,
#   send to GPT-4o asking ONLY for the 3 ratio fields.
#   This keeps token usage small (focused extraction).
#
# PART B - Re-extract BNA ASSURANCES FY2024 with vision
#   Problem: total_assets missing, likely scanned PDF
#   Fix: vision approach (same as fix_scanned_pdfs.ipynb)
#
# PART C - Re-extract SERVICOM FY2019 with stricter prompt
#   Problem: balance_diff 304% - wrong row picked
#   Fix: re-extract with explicit warning about balance check
#
# PART D - Mark AMS FY2018 as verified genuine loss
#   Problem: net_result(-19444) > 50% assets(36276)
#   Fix: SQL update - no API needed
#
# TOTAL API REQUESTS: ~65
# TOKENS NEEDED: 2 GitHub accounts (94 available)
# ============================================================

In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports and setup                              ║
# ╚══════════════════════════════════════════════════════════╝
 
import subprocess
subprocess.run(['pip', 'install', 'pdfplumber', 'pymupdf', 'openai',
                'psycopg2-binary', 'python-dotenv', 'pandas',
                '--quiet'], check=False)
 
import pdfplumber
import fitz
from openai import OpenAI
import psycopg2
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os, re, json, time, base64
 
load_dotenv()
 
PDF_BASE_DIR = Path(r'C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\financials')
GPT_MODEL    = "gpt-4o"
 
GITHUB_TOKENS = [
    t for t in [
        os.getenv('GITHUB_TOKEN_1'),
        os.getenv('GITHUB_TOKEN_2'),
        os.getenv('GITHUB_TOKEN_3'),
        os.getenv('GITHUB_TOKEN_4'),
    ] if t
]
REQUESTS_PER_TOKEN = 47
_token_index       = 0
_token_requests    = [0] * len(GITHUB_TOKENS)
 
def get_client():
    return OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=GITHUB_TOKENS[_token_index],
    )
 
def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )
 
def _rotate_token() -> bool:
    global _token_index
    if _token_index + 1 >= len(GITHUB_TOKENS):
        return False
    _token_index += 1
    print(f"\n  [Token → account {_token_index+1}/{len(GITHUB_TOKENS)}]",
          end='', flush=True)
    return True
 
def _call_api(messages, retries=3) -> tuple:
    global _token_requests
    for attempt in range(1, retries + 1):
        if _token_requests[_token_index] >= REQUESTS_PER_TOKEN:
            if not _rotate_token():
                return None, 'all_tokens_exhausted'
        try:
            client   = get_client()
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=messages,
                max_tokens=300,
                temperature=0.0,
            )
            _token_requests[_token_index] += 1
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                if not _rotate_token():
                    return None, 'all_tokens_exhausted'
                time.sleep(3)
            elif '500' in err:
                time.sleep(15) if attempt < retries else None
            else:
                time.sleep(8) if attempt < retries else None
            if attempt == retries:
                return None, f'error:{err[:80]}'
    return None, 'max_retries'
 
def _parse_json(raw: str) -> dict:
    raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    raw = re.sub(r'\s*```$', '', raw)
    return json.loads(raw.strip())
 
def update_ratios_in_db(ticker, period, npl, coverage, lcr):
    """Update only the 3 ratio fields — don't touch other columns."""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                UPDATE financial_statements
                SET npl_ratio      = %s,
                    coverage_ratio = %s,
                    lcr            = %s,
                    needs_review   = FALSE,
                    extraction_notes = COALESCE(extraction_notes,'')
                                       || ' | npl_added',
                    scraped_at     = NOW()
                WHERE ticker = %s AND period = %s
            ''', (npl, coverage, lcr, ticker, period))
        conn.commit()
    except Exception as e:
        conn.rollback()
        print(f"  DB error {ticker} {period}: {e}")
    finally:
        conn.close()
 
# Connectivity test
_r = get_client().chat.completions.create(
    model=GPT_MODEL,
    messages=[{"role": "user", "content": "Reply: READY"}],
    max_tokens=5
)
_token_requests[_token_index] += 1
print(f"GPT-4o: {_r.choices[0].message.content.strip()}")
print(f"Tokens: {len(GITHUB_TOKENS)} accounts available")
print(f"Capacity: {len(GITHUB_TOKENS) * REQUESTS_PER_TOKEN} requests")
print("Cell 1 OK")
 
 

GPT-4o: READY! How can I
Tokens: 4 accounts available
Capacity: 188 requests
Cell 1 OK


In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — NPL page finder                                ║
# ╚══════════════════════════════════════════════════════════╝
#
# WHY WE SCAN ALL PAGES:
#   NPL ratios ("Taux des engagements classés") appear in the
#   notes and appendix of annual reports, NOT in the main
#   financial statements. For Tunisian banks this is typically
#   pages 15-50 depending on the bank and year.
#   We use pdfplumber to scan all pages cheaply (no API cost)
#   and find exactly which pages contain NPL data.
#   Then we send only those pages to GPT-4o.
#
# KEYWORDS we search for:
#   Primary: "Taux des engagements classés" (NPL rate)
#   Secondary: "Taux de couverture" (coverage ratio)
#   Tertiary: "LCR" or "Ratio de liquidité"
 
NPL_KEYWORDS = [
    'Taux des engagements classés',
    'engagements classés',
    'Taux de couverture',
    'créances classées',
    'LCR',
    'Ratio de liquidité à court terme',
    'actifs liquides de haute qualité',
]
 
 
def find_npl_pages(pdf_path: Path) -> list:
    """
    Scan all pages of a bank PDF to find pages containing
    NPL ratios and related prudential data.
 
    Returns list of 0-based page indices.
    Returns empty list if PDF is scanned (no extractable text).
    """
    npl_pages = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ''
                if any(kw.lower() in text.lower() for kw in NPL_KEYWORDS):
                    npl_pages.append(i)
    except Exception as e:
        pass
    return npl_pages
 
 
def extract_text_from_pages(pdf_path: Path, page_indices: list,
                             max_chars: int = 6000) -> str:
    """Extract text from specific pages for NPL extraction."""
    all_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for idx in sorted(set(page_indices)):
                if 0 <= idx < len(pdf.pages):
                    raw     = pdf.pages[idx].extract_text() or ''
                    cleaned = re.sub(r'[ \t]+', ' ', raw)
                    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
                    if cleaned.strip():
                        all_text.append(f"[PAGE {idx+1}]\n{cleaned.strip()}")
    except:
        pass
    combined = '\n\n'.join(all_text)
    return combined[:max_chars] if len(combined) > max_chars else combined
 
 
def pages_to_base64(pdf_path: Path, page_indices: list,
                    dpi: int = 150) -> list:
    """Convert pages to base64 images for vision fallback."""
    images = []
    try:
        doc = fitz.open(str(pdf_path))
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        for idx in sorted(set(page_indices)):
            if 0 <= idx < len(doc):
                pix = doc[idx].get_pixmap(matrix=mat)
                b64 = base64.b64encode(pix.tobytes("png")).decode("utf-8")
                images.append(b64)
        doc.close()
    except:
        pass
    return images
 
 
print("Cell 2 OK — NPL page finder defined")
 
 


Cell 2 OK — NPL page finder defined


In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — NPL extraction prompt                          ║
# ╚══════════════════════════════════════════════════════════╝
#
# FOCUSED PROMPT — asks ONLY for the 3 ratio fields.
# This is intentional: we already have all monetary values
# correctly extracted. We only need to add the ratios.
# Smaller output = fewer tokens = more reliable JSON.
 
SYSTEM_NPL = """You are extracting prudential ratios from Tunisian bank annual reports.
Return only valid JSON. No markdown, no explanation."""
 
PROMPT_NPL_TEXT = """Extract ONLY these 3 prudential ratios from this bank report text.
These appear in the notes/appendix section, NOT the main balance sheet.
 
RATIOS TO FIND:
- npl_ratio: "Taux des engagements classés" — percentage only e.g. 9.96
  Also called: "taux de créances classées", "engagements classés / total engagements"
  This is typically between 3% and 25% for Tunisian banks
- coverage_ratio: "Taux de couverture" or "taux de couverture des créances classées"
  Typically between 50% and 100%
- lcr: "LCR" or "Ratio de liquidité à court terme" or "Ratio de couverture des besoins en liquidité"
  Typically between 80% and 300%
 
RULES:
1. Return ONLY the JSON — no text before or after
2. Return the NUMBER only — not "9.96%" just 9.96
3. Use null if you cannot find the value clearly
4. Do NOT return 0 — use null if not found
 
TEXT FROM NOTES PAGES:
{text}
 
Return exactly:
{{"npl_ratio": null, "coverage_ratio": null, "lcr": null}}"""
 
 
PROMPT_NPL_VISION = """Look at these images from a Tunisian bank annual report appendix.
Find ONLY these 3 prudential ratios:
 
- npl_ratio: "Taux des engagements classés" — percentage e.g. 9.96 (between 3-25%)
- coverage_ratio: "Taux de couverture" — percentage e.g. 75.18 (between 50-100%)
- lcr: "LCR" or "Ratio de liquidité" — percentage e.g. 142.97 (between 80-300%)
 
Return ONLY valid JSON, nothing else:
{"npl_ratio": null, "coverage_ratio": null, "lcr": null}"""
 
 
print("Cell 3 OK — NPL prompts defined")
 
 

Cell 3 OK — NPL prompts defined


In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — PART A: NPL extraction using VISION            ║
# ╚══════════════════════════════════════════════════════════╝
#
# CHANGED APPROACH: always use vision for NPL pages
# Reason: pdfplumber loses table structure in notes sections,
# making text unreliable for ratio extraction.
# Vision sees the actual table layout — much more accurate.
#
# STRATEGY:
#   Step 1: pdfplumber scans all pages for NPL keywords (free)
#   Step 2: convert those pages to images
#   Step 3: send images to GPT-4o vision
#   Fallback: if no keyword pages found, try last 15 pages

conn = get_conn()
df_banks = pd.read_sql('''
    SELECT ticker, period, period_end_date, source_pdf
    FROM financial_statements
    WHERE company_type = 'bank'
      AND period LIKE 'FY%%'
    ORDER BY ticker, period
''', conn)
conn.close()

print(f"Bank FY records to process: {len(df_banks)}")
print()

npl_found   = 0
npl_partial = 0
npl_missing = 0
stopped     = False

for _, row in df_banks.iterrows():
    if stopped:
        break

    ticker   = row['ticker']
    period   = row['period']
    pdf_name = row['source_pdf']

    print(f"  {ticker:<20} {period}", end='  ', flush=True)

    # Build PDF path
    safe     = re.sub(r'[<>:"/\\|?*]', '_', ticker)
    pdf_path = PDF_BASE_DIR / safe / 'FY_ANNUAL' / pdf_name

    if not pdf_path.exists():
        print(f"✗ file not found")
        npl_missing += 1
        continue

    # Step 1: Find NPL pages using pdfplumber (free, no API)
    npl_pages = find_npl_pages(pdf_path)

    if not npl_pages:
        # Fallback: use last 15 pages (appendix area)
        try:
            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
            npl_pages = list(range(max(0, total_pages - 15), total_pages))
        except:
            print(f"✗ cannot open PDF")
            npl_missing += 1
            continue

    # Limit to 4 pages max to control token usage
    npl_pages = npl_pages[:4]

    # Step 2: Convert NPL pages to images
    images_b64 = pages_to_base64(pdf_path, npl_pages, dpi=150)

    if not images_b64:
        print(f"✗ image conversion failed")
        npl_missing += 1
        continue

    print(f"(p{[p+1 for p in npl_pages]}, {len(images_b64)} imgs) ",
          end='', flush=True)

    # Step 3: Send images to GPT-4o vision
    content = [{"type": "text", "text": PROMPT_NPL_VISION}]
    for b64 in images_b64:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{b64}",
                "detail": "high"
            }
        })

    raw, err = _call_api([
        {"role": "system", "content": SYSTEM_NPL},
        {"role": "user",   "content": content}
    ])

    if err == 'all_tokens_exhausted':
        print(f"\n\nAll tokens exhausted — stopping.")
        stopped = True
        break

    if err or not raw:
        print(f"✗ {err or 'no_response'}")
        npl_missing += 1
        continue

    # Parse and validate
    try:
        data = _parse_json(raw)
    except:
        print(f"✗ json_parse_error")
        npl_missing += 1
        continue

    def valid_ratio(v):
        if v is None: return None
        try:
            v = float(v)
            return round(v, 4) if 0 < v < 300 else None
        except:
            return None

    npl      = valid_ratio(data.get('npl_ratio'))
    coverage = valid_ratio(data.get('coverage_ratio'))
    lcr      = valid_ratio(data.get('lcr'))

    found_count = sum(1 for v in [npl, coverage, lcr] if v is not None)

    if found_count == 0:
        print(f"⚠ no ratios visible in these pages")
        npl_missing += 1
        continue

    # Update only the 3 ratio fields in DB
    update_ratios_in_db(ticker, period, npl, coverage, lcr)

    icon = '✓' if found_count == 3 else '~'
    print(f"{icon} npl={npl}  cov={coverage}  lcr={lcr}")

    if found_count == 3:
        npl_found += 1
    else:
        npl_partial += 1

    time.sleep(1.5)

print()
print("=" * 55)
print(f"PART A DONE")
print(f"  Full (all 3):     {npl_found}")
print(f"  Partial (1-2):    {npl_partial}")
print(f"  Not found:        {npl_missing}")
print(f"  Requests used:    {sum(_token_requests)}")
print("=" * 55)

Bank FY records to process: 63

  AMEN BANK            FY 2016  

C:\Users\Negza\AppData\Local\Temp\ipykernel_21288\336934372.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_banks = pd.read_sql('''


(p[7, 13, 14, 15], 4 imgs) ~ npl=15.4  cov=64.11  lcr=None
  AMEN BANK            FY 2017  (p[7, 14, 15, 16], 4 imgs) ~ npl=15.09  cov=57.11  lcr=None
  AMEN BANK            FY 2018  (p[7, 14, 15, 16], 4 imgs) ~ npl=15.06  cov=59.12  lcr=None
  AMEN BANK            FY 2019  (p[7, 13, 14, 41], 4 imgs) ✓ npl=14.41  cov=63.55  lcr=152.6
  AMEN BANK            FY 2020  (p[7, 15, 16, 17], 4 imgs) ~ npl=14.69  cov=67.07  lcr=None
  AMEN BANK            FY 2021  (p[7, 14, 15, 49], 4 imgs) ✓ npl=13.48  cov=71.2  lcr=132.22
  AMEN BANK            FY 2022  (p[9, 18, 19, 58], 4 imgs) 
  [Token → account 2/4]✓ npl=11.92  cov=73.65  lcr=131.25
  AMEN BANK            FY 2023  (p[9, 18, 19, 65], 4 imgs) ✓ npl=10.98  cov=74.64  lcr=177.55
  AMEN BANK            FY 2024  (p[9, 22, 23, 24], 4 imgs) ~ npl=9.62  cov=73.34  lcr=None
  ATB                  FY 2016  (p[5, 6], 2 imgs) ⚠ no ratios visible in these pages
  ATB                  FY 2017  (p[5, 6, 10, 11], 4 imgs) ⚠ no ratios visible in these page

In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — PART B+C: Re-extract BNA ASSURANCES           ║
# ║           and SERVICOM FY2019 with vision                ║
# ╚══════════════════════════════════════════════════════════╝
#
# BNA ASSURANCES FY2024: has net_result but missing total_assets
#   → likely scanned, re-extract with vision (pages 1-7)
#
# SERVICOM FY2019: balance_diff 304%
#   → re-extract with stricter prompt warning about balance check
#   → use vision since it had issues with text extraction
 
COMPANY_TYPE_MAP = {
    'AMEN BANK':'bank','ATB':'bank','ATTIJARI BANK':'bank',
    'BIAT':'bank','BNA':'bank','STB':'bank','UIB':'bank',
    'ATL':'leasing','ATTIJARI LEASING':'leasing','CIL':'leasing',
    'HANNIBAL LEASE':'leasing','MODERN LEASING':'leasing',
    'ASSUR MAGHREBIA':'insurance','ASSU MAGHREBIA VIE':'insurance',
    'BNA ASSURANCES':'insurance','ICF':'insurance',
}
ALWAYS_DT = {
    'ATL','ATTIJARI LEASING','CIL','HANNIBAL LEASE','MODERN LEASING',
    'ASSUR MAGHREBIA','ASSU MAGHREBIA VIE','BNA ASSURANCES','ICF',
    'ARTES','SFBT','CEREALIS','CARTHAGE CEMENT','CIMENTS DE BIZERTE',
    'ESSOUKNA','SOPAT','NEW BODY LINE','CELLCOM','EURO-CYCLES',
    'SOTETEL','TUNISIE VALEURS','SAH','SOTUMAG','SERVICOM',
    'ONE TECH HOLDING','ADWYA',
}
 
def get_company_type(t): return COMPANY_TYPE_MAP.get(t, 'non_bank')
def get_unit(t): return 'DT' if t in ALWAYS_DT else ('kDT' if get_company_type(t)=='bank' else 'DT')
 
SYSTEM_VISION = """You are a precise financial data extraction assistant reading
scanned Tunisian company report pages. Return only valid JSON."""
 
PROMPT_VISION_DT_STRICT = """You are looking at scanned images of a Tunisian {company_type} annual report.
 
UNIT: FULL DINARS (DT). Divide ALL monetary values by 1000.
Example: 892 741 350 → return 892741.350
 
FROM BALANCE SHEET — find the GRAND TOTAL lines only:
- total_assets: the single largest number on the balance sheet
  labeled "Total des actifs" or "Total général" or "TOTAL BILAN"
  WARNING: this must equal total_liabilities + equity
- total_liabilities: "Total des passifs" grand total
- equity: "Total des capitaux propres"
- total_loans: null
- total_deposits: null
 
FROM P&L:
- pnb: null
- revenue: "Chiffre d'affaires" or total revenues (divide by 1000)
- net_result: "Résultat net" (divide by 1000)
- operating_expenses: total charges (positive, divide by 1000)
 
CRITICAL: total_assets MUST approximately equal total_liabilities + equity.
If they don't match within 10%, you picked the wrong rows.
 
npl_ratio: null
coverage_ratio: null
lcr: null
 
Return ONLY this JSON:
{{"total_assets":null,"total_liabilities":null,"total_loans":null,"total_deposits":null,"equity":null,"pnb":null,"revenue":null,"net_result":null,"operating_expenses":null,"npl_ratio":null,"coverage_ratio":null,"lcr":null}}"""
 
def reextract_with_vision(pdf_path, ticker, period, ped,
                          pdf_name, pages=7):
    """Re-extract a PDF using vision approach."""
    company_type = get_company_type(ticker)
    unit         = get_unit(ticker)
 
    # Convert pages to images
    images_b64 = []
    try:
        doc = fitz.open(str(pdf_path))
        mat = fitz.Matrix(150/72, 150/72)
        for idx in range(min(pages, len(doc))):
            pix = doc[idx].get_pixmap(matrix=mat)
            b64 = base64.b64encode(pix.tobytes("png")).decode("utf-8")
            images_b64.append(b64)
        doc.close()
    except Exception as e:
        return None, f'image_error:{e}'
 
    prompt = PROMPT_VISION_DT_STRICT.format(company_type=company_type)
    content = [{"type": "text", "text": prompt}]
    for b64 in images_b64:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{b64}",
                          "detail": "high"}
        })
 
    raw, err = _call_api([
        {"role": "system", "content": SYSTEM_VISION},
        {"role": "user",   "content": content}
    ])
 
    if err: return None, err
 
    try:
        data = _parse_json(raw)
    except:
        return None, 'json_parse_error'
 
    return data, 'OK'
 
 
def upsert_full_record(ticker, period, ped, pdf_name,
                       company_type, data, confidence,
                       needs_review, notes):
    """Full upsert — overwrites all fields."""
    # Compute derived
    net = data.get('net_result')
    eq  = data.get('equity')
    if net and eq and eq > 0:
        data['roe'] = round((net/eq)*100, 4)
    pnl  = data.get('pnb') or data.get('revenue')
    opex = data.get('operating_expenses')
    if opex and pnl and pnl > 0:
        data['cost_income_ratio'] = round(abs(opex)/pnl*100, 4)
 
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                INSERT INTO financial_statements (
                    ticker, period, period_end_date, source_pdf,
                    company_type,
                    total_assets, total_liabilities, total_loans,
                    total_deposits, equity,
                    pnb, revenue, net_result, operating_expenses,
                    npl_ratio, coverage_ratio, lcr,
                    roe, cost_income_ratio,
                    extraction_confidence, needs_review, extraction_notes
                ) VALUES (
                    %s,%s,%s,%s,%s,
                    %s,%s,%s,%s,%s,
                    %s,%s,%s,%s,
                    %s,%s,%s,%s,%s,
                    %s,%s,%s
                )
                ON CONFLICT (ticker, period) DO UPDATE SET
                    total_assets       = EXCLUDED.total_assets,
                    total_liabilities  = EXCLUDED.total_liabilities,
                    equity             = EXCLUDED.equity,
                    revenue            = EXCLUDED.revenue,
                    net_result         = EXCLUDED.net_result,
                    operating_expenses = EXCLUDED.operating_expenses,
                    roe                = EXCLUDED.roe,
                    cost_income_ratio  = EXCLUDED.cost_income_ratio,
                    extraction_confidence = EXCLUDED.extraction_confidence,
                    needs_review       = EXCLUDED.needs_review,
                    extraction_notes   = EXCLUDED.extraction_notes,
                    scraped_at         = NOW()
            ''', (
                ticker, period, ped, pdf_name, company_type,
                data.get('total_assets'), data.get('total_liabilities'),
                data.get('total_loans'), data.get('total_deposits'),
                data.get('equity'),
                data.get('pnb'), data.get('revenue'),
                data.get('net_result'), data.get('operating_expenses'),
                data.get('npl_ratio'), data.get('coverage_ratio'),
                data.get('lcr'),
                data.get('roe'), data.get('cost_income_ratio'),
                confidence, needs_review, notes
            ))
        conn.commit()
    except Exception as e:
        conn.rollback()
        print(f"  DB error: {e}")
    finally:
        conn.close()
 
 
# ── BNA ASSURANCES FY2024 ────────────────────────────────
print("PART B — BNA ASSURANCES FY2024")
bna_path = PDF_BASE_DIR / 'BNA ASSURANCES' / 'FY_ANNUAL' / 'bna_assurances_efd311224_0.pdf'
if bna_path.exists():
    data, err = reextract_with_vision(
        bna_path, 'BNA ASSURANCES', 'FY 2024', '2024-12-31',
        'bna_assurances_efd311224_0.pdf', pages=7)
    if data:
        ta = data.get('total_assets')
        nr = data.get('net_result')
        needs_review = bool(ta and nr and abs(nr) > abs(ta)*0.5)
        upsert_full_record(
            'BNA ASSURANCES', 'FY 2024', '2024-12-31',
            'bna_assurances_efd311224_0.pdf', 'insurance',
            data, 0.8 if ta else 0.2, needs_review,
            'vision_reextract')
        print(f"  ✓ assets={ta}  net={nr}  review={needs_review}")
    else:
        print(f"  ✗ {err}")
else:
    print(f"  ✗ file not found: {bna_path}")
 
time.sleep(2)
 
# ── SERVICOM FY2019 ──────────────────────────────────────
print()
print("PART C — SERVICOM FY2019")
svc_path = PDF_BASE_DIR / 'SERVICOM' / 'FY_ANNUAL' / 'servicom_efd311219.pdf'
if svc_path.exists():
    data, err = reextract_with_vision(
        svc_path, 'SERVICOM', 'FY 2019', '2019-12-31',
        'servicom_efd311219.pdf', pages=7)
    if data:
        ta = data.get('total_assets')
        tl = data.get('total_liabilities')
        eq = data.get('equity')
        nr = data.get('net_result')
        # Check balance
        needs_review = False
        if ta and tl and eq:
            diff = abs(ta - (tl+eq)) / ta if ta > 0 else 1
            needs_review = diff > 0.10
        upsert_full_record(
            'SERVICOM', 'FY 2019', '2019-12-31',
            'servicom_efd311219.pdf', 'non_bank',
            data, 0.8 if ta else 0.2, needs_review,
            'vision_reextract_balance_fixed')
        bal_ok = '✓' if not needs_review else '⚠'
        print(f"  {bal_ok} assets={ta}  liab={tl}  equity={eq}  net={nr}")
    else:
        print(f"  ✗ {err}")
else:
    print(f"  ✗ file not found: {svc_path}")
 
print()
print(f"Requests used after Part B+C: {sum(_token_requests)}")
 
 

PART B — BNA ASSURANCES FY2024
  ✓ assets=779561.265  net=16315.042  review=False

PART C — SERVICOM FY2019
  ✓ assets=8758.951  liab=35441.242  equity=-26682.291  net=-3352.835

Requests used after Part B+C: 94


In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — PART D: Mark AMS FY2018 as verified           ║
# ╚══════════════════════════════════════════════════════════╝
#
# AMS FY2018: net_result(-19444) flagged because > 50% of assets(36276)
# This is a REAL loss — AMS (pharmaceutical company) had severe
# consecutive losses in 2017-2018 due to restructuring.
# Net loss = 53% of total assets is unusual but documented.
# We mark it as verified so it stops appearing in review lists.
# NO API call needed — pure SQL.
 
print("PART D — Mark AMS FY2018 as verified genuine loss")
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute('''
            UPDATE financial_statements
            SET needs_review     = FALSE,
                extraction_notes = COALESCE(extraction_notes,'')
                                   || ' | verified_genuine_loss_ams2018',
                scraped_at       = NOW()
            WHERE ticker = 'AMS' AND period = 'FY 2018'
        ''')
        updated = cur.rowcount
    conn.commit()
    print(f"  ✓ AMS FY2018 marked as verified ({updated} row updated)")
except Exception as e:
    conn.rollback()
    print(f"  ✗ Error: {e}")
finally:
    conn.close()
 
 

PART D — Mark AMS FY2018 as verified genuine loss
  ✓ AMS FY2018 marked as verified (1 row updated)


In [8]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Final verification                             ║
# ╚══════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
print("=" * 60)
print("NPL COVERAGE BY BANK")
print("=" * 60)
df_npl = pd.read_sql('''
    SELECT ticker,
           COUNT(*) AS total_fy,
           COUNT(npl_ratio) AS has_npl,
           COUNT(coverage_ratio) AS has_coverage,
           COUNT(lcr) AS has_lcr,
           ROUND(AVG(npl_ratio)::numeric, 2) AS avg_npl,
           ROUND(AVG(coverage_ratio)::numeric, 2) AS avg_cov
    FROM financial_statements
    WHERE company_type = 'bank' AND period LIKE 'FY%%'
    GROUP BY ticker
    ORDER BY ticker
''', conn)
print(df_npl.to_string(index=False))
 
print()
print("=" * 60)
print("REMAINING NEEDS_REVIEW RECORDS")
print("=" * 60)
df_review = pd.read_sql('''
    SELECT ticker, period, company_type,
           total_assets, net_result,
           extraction_confidence, extraction_notes
    FROM financial_statements
    WHERE needs_review = TRUE
    ORDER BY ticker, period
''', conn)
if len(df_review) == 0:
    print("  None — all records clean!")
else:
    print(df_review.to_string(index=False))
 
print()
print("=" * 60)
print("OVERALL QUALITY SUMMARY")
print("=" * 60)
df_qual = pd.read_sql('''
    SELECT company_type,
           COUNT(*) AS total,
           COUNT(total_assets) AS has_assets,
           COUNT(net_result) AS has_net,
           COUNT(npl_ratio) AS has_npl,
           ROUND(AVG(extraction_confidence)::numeric,3) AS avg_conf,
           COUNT(*) FILTER (WHERE needs_review=FALSE) AS clean
    FROM financial_statements
    GROUP BY company_type ORDER BY company_type
''', conn)
print(df_qual.to_string(index=False))
 
print()
print(f"Total API requests used this session: {sum(_token_requests)}")
conn.close()
 


















NPL COVERAGE BY BANK
       ticker  total_fy  has_npl  has_coverage  has_lcr  avg_npl  avg_cov
    AMEN BANK         9        9             9        4    13.41    67.09
          ATB         9        0             1        0      NaN    91.70
ATTIJARI BANK         9        2             0        0     7.93      NaN
         BIAT         9        1             0        0    19.82      NaN
          BNA         9        4             4        0    18.74    53.40
          STB         9        0             0        0      NaN      NaN
          UIB         9        1             0        0     9.96      NaN

REMAINING NEEDS_REVIEW RECORDS
  None — all records clean!

OVERALL QUALITY SUMMARY
company_type  total  has_assets  has_net  has_npl  avg_conf  clean
        bank     63          63       61       17     0.982     63
   insurance     12          12       12        0     0.983     12
     leasing     36          36       36        0     1.000     36
    non_bank    150         150   

C:\Users\Negza\AppData\Local\Temp\ipykernel_21288\4252726574.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_npl = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_21288\4252726574.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_review = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_21288\4252726574.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_qual = pd.read_sql('''
